# Aggregation window — OrientationJ across the sizes of its aggregation

The structure tensor sees the image through its Gaussian window: this study
runs **OrientationJ** (the Fiji plugin, cubic-spline gradient) on 4 images chosen for
structures of several sizes —

- `synthetic_wave_512` — three superposed waves of periods 2, 8 and 64 px at
  5°, 45° and 85°, plus uniform noise,
- `synthetic_chirp_1024` — continuous frequency sweep from center to border,
- `collagen`, `cell_aemisegger` — real fibers at several widths —

and sweeps the two sizes over which it aggregates:

1. the **structure-tensor window σ** = 1, 2, 3, 4, 5, 6, 8, 10
   ([macro-aggregation-analysis.ijm](macro-aggregation-analysis.ijm) saves the
   orientation map and the color survey per σ),
2. the **vector-field grid size** = 8, 12, 16, 24, 32, 48, 64 px at σ = 1
   ([macro-aggregation-vectorfield.ijm](macro-aggregation-vectorfield.ijm) saves the
   OJ-Vector-Field table per grid).

This notebook draws, per image: the **masked distribution of orientation for every
σ** (masks of [../../test-images/masks](../../test-images/masks)), an **animated GIF
of the color survey** across σ, a **vector-field overview** across grid sizes, and
an **animated GIF of the vector field**.

**Before running this notebook**, produce the Fiji results (kept in `results/fiji/`):

```
Fiji.app/Contents/MacOS/ImageJ-macosx --headless --console -macro macro-aggregation-analysis.ijm
Fiji.app/Contents/MacOS/ImageJ-macosx -macro macro-aggregation-vectorfield.ijm quit
```

(the second one needs the GUI — the results table cannot be saved headless —
and quits Fiji by itself when done).

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colormaps
from matplotlib.collections import LineCollection
from PIL import Image
import tifffile

IMAGES = ['synthetic_wave_512', 'synthetic_chirp_1024', 'collagen',
          'cell_aemisegger']
SIGMAS = [1, 2, 3, 4, 5, 6, 8, 10]
GRIDS = [8, 12, 16, 24, 32, 48, 64]
INPUT = os.path.join('..', '..', 'test-images')
MASKS = os.path.join(INPUT, 'masks')
FIJI = os.path.join('results', 'fiji')
OUTPUT = 'results'
os.makedirs(OUTPUT, exist_ok=True)

ANGLES = np.arange(180) - 89.5
SIGMA_COLORS = colormaps['Blues'](np.linspace(0.35, 1.0, len(SIGMAS)))

plt.rcParams.update({
    'figure.facecolor': 'white', 'font.size': 10, 'axes.titlesize': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#cfcec6', 'grid.color': '#e8e8e4', 'grid.linewidth': 0.8,
    'text.color': '#3a3a37', 'axes.labelcolor': '#3a3a37',
    'xtick.color': '#6b6a63', 'ytick.color': '#6b6a63'})

images = {name: tifffile.imread(os.path.join(INPUT, 'images', name + '.tif')).astype(np.float64)
          for name in IMAGES}
masks = {name: tifffile.imread(os.path.join(MASKS, name + '.tif')) > 0
         for name in IMAGES}
expected = ([os.path.join(FIJI, 'OJ-Orientation-' + n + '-s' + str(s) + '.tif') for n in IMAGES for s in SIGMAS]
            + [os.path.join(FIJI, 'OJ-Survey-' + n + '-s' + str(s) + '.png') for n in IMAGES for s in SIGMAS]
            + [os.path.join(FIJI, 'OJ-Vectors-' + n + '-g' + str(g) + '.csv') for n in IMAGES for g in GRIDS])
found = sum(os.path.exists(f) for f in expected)
print(len(IMAGES), 'images loaded;', found, 'of', len(expected), 'Fiji result files')

## 1a — Masked distribution of orientation versus σ

One panel per image, one curve per σ (light → dark blue = small → large window),
pixels inside the mask only.

In [ ]:
def masked_density(theta_degrees, mask):
    bins = np.clip(np.floor(90.0 + theta_degrees).astype(int), 0, 179)
    counts = np.bincount(bins[mask], minlength=180).astype(float)
    return counts / counts.sum()


densities = {}
for name in IMAGES:
    for sigma in SIGMAS:
        theta = tifffile.imread(os.path.join(
            FIJI, 'OJ-Orientation-' + name + '-s' + str(sigma) + '.tif'))
        densities[(name, sigma)] = masked_density(theta.astype(np.float64),
                                                  masks[name])

fig, axes = plt.subplots(2, 2, figsize=(13.5, 7.4), sharex=True)
for ax, name in zip(axes.flat, IMAGES):
    for sigma, color in zip(SIGMAS, SIGMA_COLORS):
        ax.plot(ANGLES, densities[(name, sigma)], color=color, linewidth=1.2,
                label='σ = ' + str(sigma))
    ax.set_xlim(-90, 90)
    ax.set_xticks(range(-90, 91, 30))
    ax.set_title(name, fontsize=10)
    ax.grid(True, axis='y')
    ax.set_axisbelow(True)
for ax in axes[:, 0]:
    ax.set_ylabel('probability per bin')
for ax in axes[1, :]:
    ax.set_xlabel('orientation [deg]')
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=len(SIGMAS), frameon=False,
           bbox_to_anchor=(0.5, 1.05))
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT, 'distributions-vs-sigma.png'),
            dpi=150, bbox_inches='tight')
plt.show()

## 1b — Color survey versus σ, animated

One GIF per image cycling the OrientationJ color survey (hue = orientation,
saturation = coherency, brightness constant — the macro default) over σ = 1 … 10.
Below, a static strip with three of the windows.

In [ ]:
def render_frame(draw, aspect, label, width_inches=6.0):
    """One GIF frame: a full-bleed panel drawn by `draw(ax)`, with a label."""
    fig = plt.figure(figsize=(width_inches, width_inches * aspect), dpi=100)
    ax = fig.add_axes([0, 0, 1, 1])
    draw(ax)
    ax.axis('off')
    ax.text(0.02, 0.98, label, transform=ax.transAxes, va='top', fontsize=14,
            color='white',
            bbox=dict(facecolor='black', alpha=0.5, edgecolor='none'))
    fig.canvas.draw()
    frame = Image.frombuffer('RGBA', fig.canvas.get_width_height(),
                             fig.canvas.buffer_rgba()).convert('RGB')
    plt.close(fig)
    return frame


def save_gif(frames, path, duration=700):
    frames[0].save(path, save_all=True, append_images=frames[1:],
                   duration=duration, loop=0)
    print('written', path, '({} frames)'.format(len(frames)))

In [ ]:
surveys = {}
for name in IMAGES:
    for sigma in SIGMAS:
        surveys[(name, sigma)] = plt.imread(os.path.join(
            FIJI, 'OJ-Survey-' + name + '-s' + str(sigma) + '.png'))

for name in IMAGES:
    aspect = images[name].shape[0] / images[name].shape[1]
    frames = [render_frame(lambda ax, s=sigma: ax.imshow(surveys[(name, s)]),
                           aspect, 'σ = ' + str(sigma))
              for sigma in SIGMAS]
    save_gif(frames, os.path.join(OUTPUT, name + '-survey.gif'))

STRIP = [1, 4, 10]
fig, axes = plt.subplots(len(IMAGES), len(STRIP),
                         figsize=(11.5, 3.6 * len(IMAGES)))
for row, name in enumerate(IMAGES):
    for column, sigma in enumerate(STRIP):
        ax = axes[row, column]
        ax.imshow(surveys[(name, sigma)])
        ax.axis('off')
        if row == 0:
            ax.set_title('σ = ' + str(sigma), fontsize=11)
    axes[row, 0].text(-0.05, 0.5, name, transform=axes[row, 0].transAxes,
                      rotation=90, va='center', ha='center', fontsize=9)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT, 'surveys-strip.png'), dpi=150, bbox_inches='tight')
plt.show()

The animated surveys (open in Jupyter or a browser to see them play):

![wave](results/synthetic_wave_512-survey.gif)
![chirp](results/synthetic_chirp_1024-survey.gif)
![collagen](results/collagen-survey.gif)
![cell](results/cell_aemisegger-survey.gif)

## 2 — Vector field versus grid-cell size

The OJ-Vector-Field tables (σ = 1, spline, nematic aggregation of the cell) for grid sizes 8 … 64 px. The overview
shows four grid sizes; the GIFs cycle all seven. Segments are colored by their
orientation with the survey hue wheel.

In [ ]:
tables = {}
for name in IMAGES:
    for grid in GRIDS:
        tables[(name, grid)] = pd.read_csv(os.path.join(
            FIJI, 'OJ-Vectors-' + name + '-g' + str(grid) + '.csv'))


def vector_segments(table, grid, scale=100.0):
    """The overlay segments of the plugin (vectortype=0: constant length,
    scale in percent of the cell). The table holds DX = -dx of the drawing code."""
    radius = scale / 100.0 * grid * 0.5
    dx = -table['DX'].to_numpy()
    dy = table['DY'].to_numpy()
    x1 = np.round(table['X'] + radius * dx).astype(int)
    y1 = np.round(table['Y'] - radius * dy).astype(int)
    x2 = np.round(table['X'] - radius * dx).astype(int)
    y2 = np.round(table['Y'] + radius * dy).astype(int)
    return x1, y1, x2, y2


def draw_field(ax, name, grid, linewidth=None):
    ax.imshow(images[name], cmap='gray')
    table = tables[(name, grid)]
    x1, y1, x2, y2 = vector_segments(table, grid)
    segments = np.stack([np.column_stack([x1, y1]),
                         np.column_stack([x2, y2])], axis=1)
    hue = (table['Orientation'].to_numpy() + 90.0) / 180.0
    if linewidth is None:
        linewidth = 0.5 + grid / 40.0
    ax.add_collection(LineCollection(segments, colors=colormaps['hsv'](hue),
                                     linewidths=linewidth))
    ax.axis('off')


OVERVIEW = [8, 16, 32, 64]
fig, axes = plt.subplots(len(IMAGES), len(OVERVIEW),
                         figsize=(13.5, 3.6 * len(IMAGES)))
for row, name in enumerate(IMAGES):
    for column, grid in enumerate(OVERVIEW):
        ax = axes[row, column]
        draw_field(ax, name, grid, linewidth=0.5 + grid / 60.0)
        if row == 0:
            ax.set_title('grid = ' + str(grid) + ' px', fontsize=11)
    axes[row, 0].text(-0.05, 0.5, name, transform=axes[row, 0].transAxes,
                      rotation=90, va='center', ha='center', fontsize=9)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT, 'vectorfields-overview.png'),
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
for name in IMAGES:
    aspect = images[name].shape[0] / images[name].shape[1]
    frames = [render_frame(lambda ax, g=grid: draw_field(ax, name, g),
                           aspect, 'grid = ' + str(grid) + ' px')
              for grid in GRIDS]
    save_gif(frames, os.path.join(OUTPUT, name + '-vectors.gif'), duration=800)

The animated vector fields:

![wave](results/synthetic_wave_512-vectors.gif)
![chirp](results/synthetic_chirp_1024-vectors.gif)
![collagen](results/collagen-vectors.gif)
![cell](results/cell_aemisegger-vectors.gif)